# wandb-init-run — ex1: open a wandb run with project, name, and config

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-init-run`. Running the final beacon cell reports progress against the `Logging: wandb.init run` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.init run` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-init-run`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-init-run"
DD_SUBTOPIC = "Logging: wandb.init run"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.init(project=, name=, config=)` — quick refresher

Every wandb run starts with exactly one call to `wandb.init(...)`. It opens a new run in your project, names it (so you can find it in the dashboard), and snapshots the hyperparameter dict as `config` so the UI can group runs by hparam value.

**Three kwargs that matter for ARENA training loops:**

- `project=` — string, the wandb project name. All runs land here.
- `name=` — string, the run name. ARENA uses `args.wandb_name`; if you omit it, wandb auto-generates a silly two-word name.
- `config=` — dict (or dataclass), the snapshot of hparams. Wandb stores it so you can filter / group / compare runs by lr, batch_size, etc.

**Where to call it.** Inside `pre_training_setup` (ARENA convention) or at the very top of `train()`. Never in `__init__` — that runs once at object construction, before sweep agents have a chance to override `wandb.config`.

**Symmetric with `wandb.finish()`.** Every `wandb.init` should pair with a `wandb.finish` at the end of training (separate drill).

### Exercise 1 — open a wandb run with project, name, and config

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `wandb.init(project=, name=, config=)` to open a single run from a dataclass-style args object, verified by mocking the `wandb` module.
> Keywords: wandb, init, config, mock
> ```

**KCs targeted:** `wandb-init-project-name`, `wandb-init-config-dict`

Implement `ex1_open_wandb_run(args)`. The canonical ARENA wandb-init recipe:

1. `args` is a simple object with attributes `wandb_project: str`, `wandb_name: str`, and a few hparams (`lr: float`, `batch_size: int`, `epochs: int`).
2. Call `wandb.init(...)` with EXACTLY these kwargs:
   - `project=args.wandb_project`
   - `name=args.wandb_name`
   - `config=args` itself (wandb accepts dataclasses + plain objects; it'll snapshot their attributes).
3. Return whatever `wandb.init` returned (a run handle in real wandb; the mock's MagicMock return value here).

**The test mocks `wandb`.** Colab and the build venv may not have wandb installed. The test installs a `MagicMock()` into `sys.modules['wandb']` BEFORE you import it, then inspects `wandb.init.call_args` to verify the kwargs.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex1_open_wandb_run(args):
    """Open a wandb run named args.wandb_name in args.wandb_project."""
    raise NotImplementedError()


def _test_ex1():
    from dataclasses import dataclass

    @dataclass
    class FakeArgs:
        wandb_project: str = 'arena-resnet'
        wandb_name: str = 'baseline-run-3'
        lr: float = 1e-3
        batch_size: int = 64
        epochs: int = 3

    # Reset the mock before exercising.
    wandb.init.reset_mock()
    args = FakeArgs()
    ret = ex1_open_wandb_run(args)

    # Exactly one wandb.init call.
    assert wandb.init.call_count == 1, f'expected 1 wandb.init call, got {wandb.init.call_count}'
    ca = wandb.init.call_args
    kwargs = ca.kwargs
    assert kwargs.get('project') == 'arena-resnet', f'project kwarg wrong: {kwargs.get("project")!r}'
    assert kwargs.get('name') == 'baseline-run-3', f'name kwarg wrong: {kwargs.get("name")!r}'
    assert kwargs.get('config') is args, 'config kwarg must be the args object itself (wandb snapshots its attrs)'
    # Return value comes from wandb.init.
    assert ret is wandb.init.return_value, 'must return whatever wandb.init returns (the run handle)'

    # Second run with a different args — fresh call recorded.
    wandb.init.reset_mock()
    args2 = FakeArgs(wandb_project='arena-transformer', wandb_name='sweep-run-7', lr=5e-4)
    ex1_open_wandb_run(args2)
    assert wandb.init.call_count == 1
    k2 = wandb.init.call_args.kwargs
    assert k2['project'] == 'arena-transformer'
    assert k2['name'] == 'sweep-run-7'
    assert k2['config'] is args2
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex1_open_wandb_run(args):
    return wandb.init(
        project=args.wandb_project,
        name=args.wandb_name,
        config=args,
    )
```

**Why pass `config=args` (the object), not `config=vars(args)`.** Wandb accepts dataclasses and plain objects with `__dict__` and snapshots their attributes itself. Passing the live object means wandb sees the SAME values your training loop uses — there's no drift between what wandb thinks the hparams are and what `args.lr` actually is.

**Reset before re-exercising.** In a Jupyter session, the mock accumulates calls across cells. `wandb.init.reset_mock()` at the top of the test resets `call_count` / `call_args` to a known state — useful when you want to test exactly one invocation.

**`sys.modules.setdefault` not `=`.** If a real wandb is installed later, `setdefault` won't clobber it. The mock only fills the gap when wandb is absent.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()